# Here we want to calculate the 3'ss of K562 WT and K700E Cells

In [8]:
library(tidyverse)
library(vroom)
library(data.table)
library(future)
library(future.apply)

In [9]:
all_sample_reps <- fread("/mnt/dawnccle2/processed_data/reprocess_250221/count_normalized_v4_merged/WT_all_samples_raw_counts.csv")
all_sample_reps <- all_sample_reps %>% 
  filter(mode == "INCLUDED") %>% 
  mutate(index_offset = paste0(index, "__", offset)) 

all_event_pairs <- fread("/mnt/dawnccle2/processed_data/reprocess_250221/count_normalized_v4_merged/K562_included_all_unique_event_pairs.csv") %>% 
  select(-index) %>% 
  mutate(index_offset_major_minor = paste0(index_offset_major, "___", index_offset_minor))
print("Finished reading data")

[1] "Finished reading data"


In [10]:
# Create a function to process samples to avoid code duplication
process_samples <- function(df, all_event_pairs, samples) {
  # Pre-filter the data frames
  major_indices <- all_event_pairs$index_offset_major
  minor_indices <- all_event_pairs$index_offset_minor
  valid_major_minor_pairs <- all_event_pairs$index_offset_major_minor
  
  # Filter initial dataframe to reduce size before operations
  sample_df <- df %>% 
    filter(sample %in% samples) %>%
    filter(index_offset %in% c(major_indices, minor_indices)) %>% 
    select(-count) 
  
  # Create smaller combinations tables for major and minor separately
  # This is more memory efficient than one large expand.grid
  major_combinations <- expand.grid(
    index_offset = unique(intersect(sample_df$index_offset, major_indices)),
    sample = samples,
    stringsAsFactors = FALSE
  )

  # Process major indices
  sample_df_major <- major_combinations %>%
    left_join(sample_df, by = c("index_offset", "sample")) %>%
    mutate(count_scaled = coalesce(count_scaled, 0)) %>% 
    separate(index_offset, into = c("index", "offset"), sep = "__", remove = FALSE)
  
  # Clean up to free memory
  rm(major_combinations)
  gc()
  
    
  minor_combinations <- expand.grid(
    index_offset = unique(intersect(sample_df$index_offset, minor_indices)),
    sample = samples,
    stringsAsFactors = FALSE
  )

  # Process minor indices
  sample_df_minor <- minor_combinations %>%
    left_join(sample_df, by = c("index_offset", "sample")) %>%
    select(-condition) %>%
    mutate(count_scaled = coalesce(count_scaled, 0)) %>% 
    separate(index_offset, into = c("index", "offset"), sep = "__", remove = FALSE)
  
  # Clean up to free memory
  rm(minor_combinations, sample_df)
  gc()
  
  # Join and filter
  sample_df_with_counts <- sample_df_major %>% 
    inner_join(sample_df_minor, 
              by = c("index" = "index", "sample" = "sample"),
              relationship = "many-to-many") %>% 
    mutate(index_offset_major_minor = paste0(index_offset.x, "___", index_offset.y)) %>% 
    filter(index_offset_major_minor %in% valid_major_minor_pairs)
  
  return(sample_df_with_counts)
}

In [11]:
unique_samples_condition1 <- all_sample_reps %>% 
    filter(condition == "K562WT") %>% 
    pull(sample) %>% 
    unique()

unique_samples_condition2 <- all_sample_reps %>% 
    filter(condition == "K562K700E") %>% 
    pull(sample) %>% 
    unique()
  
  
  condition1_df <- process_samples(all_sample_reps, all_event_pairs, unique_samples_condition1)
  condition2_df <- process_samples(all_sample_reps, all_event_pairs, unique_samples_condition2)
  
  condition1_df_summary <- condition1_df %>% 
    group_by(index, index_offset_major_minor) %>% 
    summarise(count_scaled.x = sum(count_scaled.x), count_scaled.y = sum(count_scaled.y)) %>% 
    mutate(total_count = count_scaled.x + count_scaled.y) %>% 
    mutate(altSS_1 = count_scaled.x/(count_scaled.x + count_scaled.y)) %>% 
    mutate(count_scaled.x = count_scaled.x/total_count * 1000) %>% 
    mutate(count_scaled.y = count_scaled.y/total_count * 1000) %>% 
    mutate(altSS_1_ratio = log2((count_scaled.x + 1)/(count_scaled.y + 1))) %>% 
    ungroup()
  
  condition2_df_summary <- condition2_df %>% 
    group_by(index_offset_major_minor) %>% 
    summarise(count_scaled.x = sum(count_scaled.x), count_scaled.y = sum(count_scaled.y)) %>% 
    mutate(total_count = count_scaled.x + count_scaled.y) %>% 
    mutate(altSS_2 = (count_scaled.x)/(count_scaled.x + count_scaled.y)) %>% 
    mutate(count_scaled.x = count_scaled.x/total_count * 1000) %>% 
    mutate(count_scaled.y = count_scaled.y/total_count * 1000) %>% 
    mutate(altSS_2_ratio = log2((count_scaled.x + 1)/(count_scaled.y + 1))) %>% 
    ungroup()

  condition1_2_merged <- merge(condition1_df_summary, condition2_df_summary, by = c("index_offset_major_minor")) %>% 
    mutate(altSS_diff = abs(altSS_1 - altSS_2))

`summarise()` has grouped output by 'index'. You can override using the
`.groups` argument.


In [12]:
# condition1_2_merged %>% arrange(desc(abs(altSS_diff)))

to_output_v1 <- condition1_2_merged %>% filter(total_count.x > 20 & total_count.y > 20) %>% arrange(desc(abs(altSS_diff)))
write_csv(to_output_v1, "/mnt/dawnccle2/processed_data/reprocess_250221/count_normalized_v4_merged/K700E_altSS/K700E_altSS_shortlisted_whitelist_events.csv")

## Restart
This version is considering all events.

In [13]:
# Get the events in the K562WT and K562K700E conditions
K562WT_sample_reps <- all_sample_reps %>% 
	filter(condition == "K562WT") %>% 
	group_by(index, index_offset, condition) %>%
	summarise(count_scaled = sum(count_scaled))

K562K700E_sample_reps <- all_sample_reps %>% 
	filter(condition == "K562K700E") %>% 
	group_by(index, index_offset, condition) %>%
	summarise(count_scaled = sum(count_scaled))

`summarise()` has grouped output by 'index', 'index_offset'. You can override
using the `.groups` argument.
`summarise()` has grouped output by 'index', 'index_offset'. You can override
using the `.groups` argument.


In [14]:
# Get the major event for each of the K562WT index based on count_scaled. 
K562WT_major_events <- K562WT_sample_reps %>% 
	group_by(index) %>% 
	slice_max(count_scaled, n = 1) %>% 
	ungroup()

major_events_list <- K562WT_major_events$index_offset
major_events_df <- data.frame(index_offset = major_events_list)


K562WT_minor_events <- K562WT_sample_reps %>% 
	filter(!index_offset %in% major_events_list) 

K562K700E_minor_events <- K562K700E_sample_reps %>% 
	filter(!index_offset %in% major_events_list) 

# Make a major events for K562K700E. If there is no major event then make a row and set the count_scaled to 0. 
K562K700E_major_events <- major_events_df %>% 
	left_join(K562K700E_sample_reps, by = "index_offset") %>% 
	mutate(count_scaled = ifelse(is.na(count_scaled), 0, count_scaled)) %>% 
	mutate(index = str_split(index_offset, "__", simplify = TRUE)[,1]) %>% 
	mutate(condition = "K562K700E")


# Make a minor events for K562WT and K562K700E. 
all_minor_events_list <- unique(c(K562WT_minor_events$index_offset, K562K700E_minor_events$index_offset))
all_minor_events_df <- data.frame(index_offset = all_minor_events_list)

K562WT_minor_events <- all_minor_events_df %>% 
	left_join(K562WT_sample_reps, by = "index_offset") %>% 
	mutate(count_scaled = ifelse(is.na(count_scaled), 0, count_scaled)) %>% 
	mutate(index = str_split(index_offset, "__", simplify = TRUE)[,1]) %>% 
	mutate(condition = "K562WT")


K562K700E_minor_events <- all_minor_events_df %>% 
	left_join(K562K700E_sample_reps, by = "index_offset") %>% 
	mutate(count_scaled = ifelse(is.na(count_scaled), 0, count_scaled)) %>% 
	mutate(index = str_split(index_offset, "__", simplify = TRUE)[,1]) %>% 
	mutate(condition = "K562K700E")

In [15]:
# Merge the major and minor events for K562WT and K562K700E. 
K562WT_altSS_merged <- K562WT_major_events %>% 
	left_join(K562WT_minor_events, by = c("index", "condition"), relationship = "many-to-many") %>% 
	mutate(index_offset_major_minor = paste0(index_offset.x, "___", index_offset.y))

K562K700E_altSS_merged <- K562K700E_major_events %>% 
	left_join(K562K700E_minor_events, by = c("index", "condition"), relationship = "many-to-many") %>% 
	mutate(index_offset_major_minor = paste0(index_offset.x, "___", index_offset.y))


In [16]:
# Merge the 2 tables by index_offset_major_minor. Do an outer join. 
compare_altSS_merged <- K562WT_altSS_merged %>% 
	full_join(K562K700E_altSS_merged, by = "index_offset_major_minor") %>%
	filter(!is.na(index_offset.y.x) & !is.na(index_offset.y.y)) %>% 
	mutate(altSS_K562WT = count_scaled.x.x/(count_scaled.x.x + count_scaled.y.x)) %>% 
	mutate(altSS_K562K700E = count_scaled.x.y/(count_scaled.x.y + count_scaled.y.y)) %>% 
	mutate(altSS_diff = abs(altSS_K562WT - altSS_K562K700E)) 



In [17]:
# Rank by altSS_diff, and for each index.x, keep the row with the biggest abs altSS_diff. 
compare_altSS_merged_filtered_top_event <- compare_altSS_merged %>% 
	mutate(count_sum.x = count_scaled.x.x + count_scaled.y.x) %>% 
	mutate(count_sum.y = count_scaled.x.y + count_scaled.y.y) %>%
	filter(count_sum.x > 20 & count_sum.y > 20) %>% 
	group_by(index.x) %>% 
	slice_max(abs(altSS_diff), n = 1) %>% 
	ungroup()



In [18]:
# Write this table to file. 
write_csv(compare_altSS_merged_filtered_top_event, "/mnt/dawnccle2/processed_data/reprocess_250221/count_normalized_v4_merged/K700E_altSS/K700E_altSS_PSI_events.csv")

In [19]:
# Now also convert it to log ratio
with_ratio_scaled <- compare_altSS_merged_filtered_top_event %>% 
	mutate(count_scaled.x.x = count_scaled.x.x/count_sum.x * 1000) %>% 
	mutate(count_scaled.x.y = count_scaled.x.y/count_sum.y * 1000) %>% 
	mutate(count_scaled.y.x = count_scaled.y.x/count_sum.x * 1000) %>% 
	mutate(count_scaled.y.y = count_scaled.y.y/count_sum.y * 1000) %>% 
	mutate(altSS_K562WT = log2((count_scaled.x.x + 1)/(count_scaled.y.x + 1))) %>% 
	mutate(altSS_K562K700E = log2((count_scaled.x.y + 1)/(count_scaled.y.y + 1)))

# Write this table to file. 
write_csv(with_ratio_scaled, "/mnt/dawnccle2/processed_data/reprocess_250221/count_normalized_v4_merged/K700E_altSS/K700E_altSS_log_ratio_events.csv")